# Геолокация снимков · эксперимент V3

Улучшаем предыдущую модель с географическими ячейками и CatBoost. Цель — минимальный RMSE геодезических расстояний. MAE — среднее расстояние ошибки, RMSE — корень из среднего квадрата расстояния. Обе метрики в километрах по эллипсоиду WGS84.

![Схема вложенной кросс-валидации и финального теста V3](docs/workflow-v3.svg)

## 1. Среда и данные

Используйте параметры предыдущего [ноутбука V2](train250k_final.ipynb) и обновлённый ZIP проекта. В Colab выберите GPU. Если новая версия проекта распакована в `/content/train250k-geo-ml`, укажите путь к данным на Диске. В новом сеансе загрузите ZIP через панель **Файлы** и задайте `ZIP_PATH`.

Используйте прежний каталог `GEO_CV_ROOT`, чтобы сохранить 37 500 отложенных снимков и переиспользовать совпадающие прогнозы V2. Новые артефакты сохраняются в его подкаталоге `v3`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from IPython.display import SVG, display

REPO_DIR = "/content/train250k-geo-ml"  # @param {type:"string"}
ZIP_PATH = ""  # @param {type:"string"}
TRAIN_250K_DIR = "/content/drive/MyDrive/train_250k"  # @param {type:"string"}
GEO_CV_ROOT = "/content/train250k-cv"  # @param {type:"string"}
INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}

checkout = Path(REPO_DIR).expanduser()
here = Path.cwd()
repo = next((p for p in (checkout, here, here.parent) if (p / "src" / "run_v3.py").is_file()), None)
if repo is None:
    if not ZIP_PATH:
        raise FileNotFoundError("Укажите ZIP_PATH или путь к новой версии проекта в REPO_DIR.")
    if checkout.exists():
        raise FileExistsError("Для нового ZIP укажите свободный путь в REPO_DIR.")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=checkout.parent) as tmp:
        with ZipFile(Path(ZIP_PATH).expanduser()) as archive:
            archive.extractall(tmp)
        roots = [Path(tmp), *Path(tmp).iterdir()]
        source = next((p for p in roots if (p / "src" / "run_v3.py").is_file()), None)
        if source is None:
            raise FileNotFoundError("В ZIP отсутствует src/run_v3.py.")
        from shutil import copytree
        copytree(source, checkout)
    repo = checkout

data, work = (Path(p).expanduser().resolve() for p in (TRAIN_250K_DIR, GEO_CV_ROOT))
env = {**os.environ, "TRAIN_250K_DIR": str(data), "GEO_CV_ROOT": str(work)}
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Выберите среду Colab с GPU.")
print(torch.cuda.get_device_name())

def run_src(script, *args):
    subprocess.run([sys.executable, "-u", str(repo / "src" / script), *args], cwd=repo, env=env, check=True)

if (repo / "docs" / "workflow-v3.svg").is_file():
    display(SVG(filename=str(repo / "docs" / "workflow-v3.svg")))

In [ ]:
if not (work / "split.json").is_file():
    if (data.is_relative_to("/content/drive") or work.is_relative_to("/content/drive")) and not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    run_src("prepare.py")
split = json.loads((work / "split.json").read_text())
assert split["trainval"] == 212_500 and split["test"] == 37_500
assert split["fold_sizes"] == [42_500] * 5
print("Разбиение подтверждено", split["fold_sizes"], "тест", split["test"])

## 2. Сравнение вариантов

1. **Температура softmax**. Проверяем 0,6, 0,8, 1,0, 1,2 и 1,6. Температура меняет концентрацию вероятностей географических ячеек.
2. **Многомасштабный ансамбль**. Объединяем две модели с 512 ячейками и одну с 1 024 ячейками. Разные начальные разбиения и масштабы снижают зависимость от конкретных границ.
3. **Локальный поиск**. Взвешиваем соседей по сходству эмбеддингов, апостериорной вероятности региона и его плотности. Сравниваем среднюю координату с прогнозом из наиболее согласованной области.
4. **Адаптивный CatBoost**. Признаки — прогнозы экспертов, уверенность, энтропия, плотность, согласие и PCA эмбеддингов. Два варианта учатся взвешивать экспертов по квадрату геодезической ошибки или лучшей комбинации пары на сферической дуге.
5. **Касательный остаток**. CatBoost корректирует сферическую среднюю экспертов. Длина коррекции ограничена 1 700 км.

Сравнение идёт на пяти внешних фолдах по 42 500 снимков. Для CatBoost внутри 170 000 обучающих снимков строим три внутренних фолда. Каждая строка стека получает прогнозы от базовых моделей, которые её исключали. Сетки, индекс поиска и PCA строятся на соответствующих обучающих данных, без меток внешней валидации.

In [ ]:
import pandas as pd

run_src("run_v3.py", "cv")
cv = json.loads((work / "v3" / "cv-summary.json").read_text())
table = (pd.DataFrame(cv["candidates"])
         .loc[:, ["name", "cv_mae_km", "cv_rmse_km"]]
         .rename(columns={"name": "Вариант", "cv_mae_km": "MAE, км", "cv_rmse_km": "RMSE, км"})
         .sort_values("RMSE, км"))
print("Победитель по пяти фолдам", cv["winner"])
display(table.round(1))

## 3. Переобучение и итоговый тест

После выбора по RMSE конфигурацию фиксируем. Базовые модели переобучаются на всех 212 500 снимках разработки, а при победе стека CatBoost получает прогнозы, сохранённые для пяти внешних валидационных частей. Затем единожды вычисляем метрики на 37 500 отложенных снимках.

Агрегаты предыдущего эксперимента на этом же тесте используются как исторический ориентир. Подбор новой конфигурации использует только пять фолдов из 85%.

In [ ]:
run_src("run_v3.py", "final")
result = json.loads((work / "v3" / "results.json").read_text())
assert result["winner"] == cv["winner"]
test = result["test"]
assert test["n"] == 37_500
display(pd.DataFrame([{"Модель": result["winner"], "Снимков": test["n"],
                       "MAE, км": test["mae_km"], "RMSE, км": test["rmse_km"]}]).round(1))
if result.get("historical_reference"):
    print("Исторический ориентир", result["historical_reference"])

In [ ]:
# Для победителя-классификатора покажем вероятности на одном снимке из 85%.
# Этот пример иллюстрирует модель и не используется для выбора гиперпараметров.
checkpoint_path = work / "v3" / "best-cells.pt"
if result["winner"].startswith("cell_") and checkpoint_path.is_file():
    import numpy as np
    import matplotlib.pyplot as plt
    sys.path.insert(0, str(repo / "src"))
    from cell_v2 import CellClassifier
    from experiment import decode

    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = CellClassifier(1280, ckpt["k"]).to("cuda")
    model.load_state_dict(ckpt["state_dict"]); model.eval()
    x = np.load(work / "dev_x.npy", mmap_mode="r")[:1].copy()
    with torch.inference_mode():
        logits = model(torch.tensor(x, device="cuda") * ckpt["input_scale"])[0].cpu()
        posterior = logits.masked_fill(ckpt["cluster_counts"] == 0, -torch.inf).softmax(dim=0)
    lat, lon = decode(ckpt["centers"].numpy())
    prior = ckpt["prior_probs"].numpy()
    prob = posterior.numpy()
    top = np.argsort(prob)[-8:][::-1]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.7), layout="constrained")
    axes[0].scatter(lon, lat, c=prob, s=8 + 420 * prob, cmap="viridis")
    axes[0].set(xlim=(-180,180), ylim=(-90,90), title="Вероятности географических ячеек",
                xlabel="Долгота, °", ylabel="Широта, °")
    axes[1].bar(np.arange(len(top))-.2, prob[top], .4, label="По эмбеддингам")
    axes[1].bar(np.arange(len(top))+.2, prior[top], .4, label="По частотам")
    axes[1].set(xticks=np.arange(len(top)), xticklabels=top, xlabel="ID ячейки",
                ylabel="Вероятность", title="Восемь наиболее вероятных ячеек")
    axes[1].legend()
    plt.show()